## 04 - Recovering Depth 

## Imports

In [1]:
import numpy as np

## A Depth Problem

We previously covered how perspective projection, when leveraging camera intrinsics and extrinsics, can allow us to transform 3D points from the real world into 2D points on an image. 

Mapping from 3D to 2D represents an inherent problem with perspective projection: we lose the depth dimension (Z), which means that two objects that may be at different points along that Z axis will be mapped to the same $(x, y)$ coordinates. 

Let's illustrate this below. 

### Two Points - Different Z, Same (X, Y)


Recall our perspective projection equations 

$x = f\frac{X}{Z}$, $y = f\frac{Y}{Z}$

and suppose we have two points in the 3D camera coordinate plane $(X_1, Y_1, Z_1)$ and $(X_2, Y_2, Z_2)$ (we omit the $c$ subscript for brevity). 

It follows that these two points will map to the same $(x, y)$ if the following relationships hold true: 

$$ \frac{X_1}{Z_1} = \frac{X_2}{Z_2}, \frac{Y_1}{Z_1} = \frac{Y_2}{Z_2} $$




But how does this happen? Don't we have our $Z_c$ term from our intrinsic matrix? 

Well, we *did*, before we did our intrinsic multiplication. 

When we found the result of the intrinsic matrix multiplication 

$$
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
= 
\begin{bmatrix}
\frac{X_c}{Z_c} \\
\frac{Y_c}{Z_c}\\
1
\end{bmatrix}
$$

we lose our $Z_c$ value, and we cannot get this back. This is due to the property of a non-invertible transformation (again another matrix property you shouldn't worry about too much for now), so if we did try to reverse the intrinsic mulitplication from the previous notebook, we would get this equation 

$$
\begin{bmatrix}
X_c \\
Y_c \\
Z_c
\end{bmatrix}
= 
\lambda
K^{-1}
p
$$

where we are again left with our unknown $\lambda$ as our depth term. 

Fortunately, there are some solutions - let's go over them. The $-1$ for our $K$ represents a matrix inverse. 


### Recovering Depth: Homography with a Planar Constraint

Recall from our homography unit that the key assumption is that our 2D images must lie on the same 3D plane, such that we can get at least 4 points in each image that capture the same object. 

This assumption can help us recover depth, up to a point. Since our Z is constant given the above contrainst, we know the depth of objects - if they lie on this plane. We find this by performing some operations on our homography matrix $H$, known as a decomposition. 

If they don't lie on the same plane (and we cannot get those 4 points), then we are not able to recover depth. Therefore, we don't typically use this approach much in object detection scenarios. 

### Recovering Depth: Triangulation w/ 2 Cameras 

#### Single Camera View

Remember our 3D perspective projection 

$$p = \lambda K MP$$

As mentioned, we have this constraint of not being able to solve for $\lambda$. You can imagine that this situation looks like this: 

![](./ref_imgs/single_camera.png)

Again, as we've talked about, we have no way of finding the depth of the object because we just have a ray. However, we do know that we can take a ray, and if we add two other rays that intersect, we get a triangle. 

How do we create this triangle? 

#### Two Camera View